# ISRO Challenge 4: Route Resilience Training Pipeline

**Instructions:**
1. **CRITICAL:** You must add the actual dataset, NOT a notebook. On the right panel under 'Add Input', search for `balraj98/deepglobe-road-extraction-dataset` (it should have a 'Dataset' tag, not a 'Notebook' tag).
2. Under **Session Options**, set Accelerator to **GPU T4x2** or **P100**.
3. Run all cells. The trained weights will be saved to your `/kaggle/working/` directory.

In [1]:
import os
import cv2
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


### 1. Model Architecture (Deep CNN + Transformer + U-Net + Skips)

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualBlock(nn.Module):
    """
    Standard ResNet-style residual block.
    Helps in training very deep networks by avoiding the vanishing gradient problem.
    """
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        residual = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += residual
        out = self.relu(out)
        
        return out

class DeepCNNBackbone(nn.Module):
    """
    A very deep ResNet-style CNN backbone to extract rich local features.
    It produces hierarchical feature maps for the U-Net skip connections.
    """
    def __init__(self, in_channels=3, base_ch=64):
        super().__init__()
        # Initial stem convolution (downsamples by 2)
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, base_ch, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(base_ch),
            nn.ReLU(inplace=True)
        )
        
        # Deep residual stages
        # Stage 1: 3 blocks, /2 resolution, 64 channels
        self.stage1 = self._make_layer(base_ch, base_ch, blocks=3, stride=1)
        # Stage 2: 4 blocks, /4 resolution, 128 channels
        self.stage2 = self._make_layer(base_ch, base_ch * 2, blocks=4, stride=2)
        # Stage 3: 6 blocks, /8 resolution, 256 channels
        self.stage3 = self._make_layer(base_ch * 2, base_ch * 4, blocks=6, stride=2)
        # Stage 4: 3 blocks, /16 resolution, 512 channels
        self.stage4 = self._make_layer(base_ch * 4, base_ch * 8, blocks=3, stride=2)

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = []
        layers.append(ResidualBlock(in_ch, out_ch, stride))
        for _ in range(1, blocks):
            layers.append(ResidualBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        s0 = self.stem(x)         # /2 resolution
        s1 = self.stage1(s0)      # /2 resolution
        s2 = self.stage2(s1)      # /4 resolution
        s3 = self.stage3(s2)      # /8 resolution
        s4 = self.stage4(s3)      # /16 resolution
        return s0, s1, s2, s3, s4

class TransformerBottleneck(nn.Module):
    """
    Applies global self-attention across the lowest-resolution feature map.
    This allows the model to "reason" globally. If a road segment is obscured
    by trees, it can attend to the visible road on either side of the trees
    to predict continuity.
    """
    def __init__(self, dim, feat_size, num_layers=8, num_heads=8, mlp_ratio=4):
        super().__init__()
        self.dim = dim
        self.h, self.w = feat_size

        # Learnable 2D positional embedding to retain spatial awareness
        self.pos_embed = nn.Parameter(torch.zeros(1, dim, self.h, self.w))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=num_heads,
            dim_feedforward=dim * mlp_ratio,
            dropout=0.1,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x: (B, C, H, W)
        b, c, h, w = x.shape
        
        # Dynamically interpolate pos_embed if input resolution differs from initialization
        if h != self.pos_embed.shape[2] or w != self.pos_embed.shape[3]:
            pos_emb = F.interpolate(self.pos_embed, size=(h, w), mode='bilinear', align_corners=False)
        else:
            pos_emb = self.pos_embed
            
        x = x + pos_emb
        tokens = x.flatten(2).transpose(1, 2)  # (B, H*W, C)
        
        # Apply Self-Attention
        tokens = self.encoder(tokens)
        tokens = self.norm(tokens)
        
        # Reshape back to spatial feature map
        out = tokens.transpose(1, 2).reshape(b, c, h, w)
        return out

class DecoderBlock(nn.Module):
    """
    Upsamples the previous feature map, concatenates with the CNN skip connection,
    and applies residual blocks to reconstruct high-resolution details.
    """
    def __init__(self, in_channels, skip_channels, out_channels, global_skip_channels=0):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        # Using residual blocks in decoder for better information flow
        self.conv1 = ResidualBlock(in_channels + skip_channels + global_skip_channels, out_channels)
        self.conv2 = ResidualBlock(out_channels, out_channels)

    def forward(self, x, skip, global_skip=None):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=True)
        
        concat_list = [x, skip]
        
        if global_skip is not None:
            if global_skip.shape[-2:] != skip.shape[-2:]:
                global_skip_resized = F.interpolate(global_skip, size=skip.shape[-2:], mode="bilinear", align_corners=True)
            else:
                global_skip_resized = global_skip
            concat_list.append(global_skip_resized)
            
        # Skip connection from CNN Backbone + Optional Transformer Global Context
        x = torch.cat(concat_list, dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class FinalUpBlock(nn.Module):
    """
    Final upsampling to reach the exact original input resolution.
    """
    def __init__(self, in_channels, global_skip_channels, out_channels):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv1 = ResidualBlock(in_channels + global_skip_channels, out_channels)
        self.conv2 = ResidualBlock(out_channels, out_channels)
        
    def forward(self, x, global_skip):
        x = self.up(x)
        if global_skip is not None:
            global_skip_resized = F.interpolate(global_skip, size=x.shape[-2:], mode="bilinear", align_corners=True)
            x = torch.cat([x, global_skip_resized], dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        return x

class DeepRoadSegModel(nn.Module):
    """
    End-to-End Pipeline:
    Input Satellite Image -> Deep CNN -> Transformer -> Deep U-Net Decoder -> Road Mask
    """
    def __init__(self, in_channels=3, base_ch=64, input_size=(256, 256), num_transformer_layers=12, num_heads=8):
        super().__init__()
        # 1. Deep Feature Extractor
        self.backbone = DeepCNNBackbone(in_channels, base_ch)

        # 2. Transformer Bottleneck
        bottleneck_dim = base_ch * 8
        bottleneck_size = (input_size[0] // 16, input_size[1] // 16)
        
        self.transformer = TransformerBottleneck(
            dim=bottleneck_dim,
            feat_size=bottleneck_size,
            num_layers=num_transformer_layers,
            num_heads=num_heads,
        )

        # 3. U-Net Decoder with Skip Connections AND Global Transformer Skips
        self.up4 = DecoderBlock(base_ch * 8, base_ch * 4, base_ch * 4)  # /16 -> /8 (concats with s3)
        self.up3 = DecoderBlock(base_ch * 4, base_ch * 2, base_ch * 2, global_skip_channels=bottleneck_dim)  # /8 -> /4 (concats with s2 and global)
        self.up2 = DecoderBlock(base_ch * 2, base_ch, base_ch, global_skip_channels=bottleneck_dim)          # /4 -> /2 (concats with s1 and global)
        
        # Final upsample block back to full resolution
        self.up1 = FinalUpBlock(base_ch, bottleneck_dim, base_ch // 2)

        # Output head
        self.head = nn.Conv2d(base_ch // 2, 1, kernel_size=1)

    def forward(self, x, return_logits=False):
        # Extract local features
        s0, s1, s2, s3, s4 = self.backbone(x)

        # Apply global reasoning for hidden/occluded roads
        bottleneck = self.transformer(s4)   

        # Decode and recover fine spatial details (edges, boundaries)
        # We inject the global 'bottleneck' context at every single scale!
        d4 = self.up4(bottleneck, s3)
        d3 = self.up3(d4, s2, global_skip=bottleneck)
        d2 = self.up2(d3, s1, global_skip=bottleneck)
        
        # Upsample to native image size
        d1 = self.up1(d2, global_skip=bottleneck)

        logits = self.head(d1)
        
        # Guarantee it matches the exact input resolution
        if logits.shape[-2:] != x.shape[-2:]:
            logits = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=True)

        if return_logits:
            return logits
            
        # Return probability map [0, 1]
        return torch.sigmoid(logits)

if __name__ == "__main__":
    # Example usage / Sanity check
    input_resolution = (256, 256)
    model = DeepRoadSegModel(in_channels=3, base_ch=64, input_size=input_resolution, num_transformer_layers=8)
    
    dummy_image = torch.randn(2, 3, 256, 256)
    prob_map = model(dummy_image)
    
    print("--- Deep CNN + Transformer + U-Net Road Segmentation ---")
    print(f"Input image shape:   {dummy_image.shape}")
    print(f"Output mask shape:   {prob_map.shape}")
    print("Output probabilities: [{:.4f}, {:.4f}]".format(prob_map.min().item(), prob_map.max().item()))
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Total model parameters: {n_params:,}")

class SoftSkeletonize(nn.Module):
    def __init__(self, iter_=5):
        super(SoftSkeletonize, self).__init__()
        self.iter = iter_

    def soft_erode(self, img):
        p1 = -F.max_pool2d(-img, (3,1), (1,1), (1,0))
        p2 = -F.max_pool2d(-img, (1,3), (1,1), (0,1))
        return torch.min(p1, p2)

    def soft_dilate(self, img):
        return F.max_pool2d(img, (3,3), (1,1), (1,1))

    def soft_open(self, img):
        return self.soft_dilate(self.soft_erode(img))

    def forward(self, img):
        img1 = img
        skel = torch.zeros_like(img)
        for i in range(self.iter):
            eroded = self.soft_erode(img1)
            opened = self.soft_open(eroded)
            skel = torch.max(skel, eroded - opened)
            img1 = eroded
        return skel

class SoftCLDiceLoss(nn.Module):
    def __init__(self, iter_=10, smooth=1.):
        super(SoftCLDiceLoss, self).__init__()
        self.iter = iter_
        self.smooth = smooth
        self.skeletonize = SoftSkeletonize(iter_)

    def forward(self, y_pred, y_true):
        skel_pred = self.skeletonize(y_pred)
        skel_true = self.skeletonize(y_true)
        # Calculate intersection and sums across batch for stability
        tprec = (torch.sum(skel_pred * y_true) + self.smooth) / (torch.sum(skel_pred) + self.smooth)
        tsens = (torch.sum(skel_true * y_pred) + self.smooth) / (torch.sum(skel_true) + self.smooth)
        cl_dice = 2.0 * (tprec * tsens) / (tprec + tsens)
        return 1.0 - cl_dice

class BCEDiceCLDiceLoss(nn.Module):
    """
    Heavily penalizes broken roads by placing a massive weight on the Topological (clDice) loss.
    """
    def __init__(self, bce_weight=0.1, dice_weight=0.2, cldice_weight=0.7):
        super().__init__()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
        self.cldice_weight = cldice_weight
        self.bce = nn.BCEWithLogitsLoss()
        self.cldice = SoftCLDiceLoss(iter_=10)

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        
        probs = torch.sigmoid(logits)
        smooth = 1e-6
        intersection = (probs * targets).sum(dim=(2, 3))
        union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
        dice_loss = 1 - (2. * intersection + smooth) / (union + smooth)
        
        cldice_loss = self.cldice(probs, targets)
        
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss.mean() + self.cldice_weight * cldice_loss


--- Deep CNN + Transformer + U-Net Road Segmentation ---
Input image shape:   torch.Size([2, 3, 256, 256])
Output mask shape:   torch.Size([2, 1, 256, 256])
Output probabilities: [0.0935, 0.9739]
Total model parameters: 52,738,593


### 2. Dataset and Augmentation Pipeline

In [3]:
class DeepGlobeDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        # Use recursive globbing to find all images
        all_files = glob.glob(os.path.join(data_dir, '**', '*.*'), recursive=True)
        potential_images = [f for f in all_files if f.lower().endswith(('.jpg', '.jpeg', '.png')) and 'mask' not in os.path.basename(f).lower()]
        
        self.images = []
        # Only add an image to the dataset if we can strictly find its corresponding mask on disk!
        for img_path in potential_images:
            img_name = os.path.basename(img_path)
            if 'sat.jpg' in img_name:
                mask_path = img_path.replace('sat.jpg', 'mask.png')
            elif '.jpg' in img_name:
                mask_path = img_path.replace('.jpg', '_mask.png')
            else:
                mask_path = img_path.replace('.png', '_mask.png')
                
            if os.path.exists(mask_path):
                self.images.append(img_path)
                
        if len(self.images) == 0:
            raise ValueError(f"No valid image-mask pairs found recursively in {data_dir}. Check directory structure!")

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        img_name = os.path.basename(img_path)
        
        if 'sat.jpg' in img_name:
            mask_path = img_path.replace('sat.jpg', 'mask.png')
        elif '.jpg' in img_name:
            mask_path = img_path.replace('.jpg', '_mask.png')
        else:
            mask_path = img_path.replace('.png', '_mask.png')
            
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            mask = mask.unsqueeze(0)
        else:
            mask = np.expand_dims(mask, axis=-1)
            
        return image, mask

def get_train_transforms(img_size=256):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.7),
        A.CoarseDropout(num_holes_range=(2, 10), hole_height_range=(10, 40), hole_width_range=(10, 40), p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

### 3. Training Loop (Saves Best Model Only)

In [4]:
# Intelligent Auto-Detection of the Training Folder
TRAIN_DIR = None

for root, dirs, files in os.walk('/kaggle/input'):
    if 'train' in dirs:
        train_path = os.path.join(root, 'train')
        if any(f.endswith('.jpg') or f.endswith('.png') for f in os.listdir(train_path)):
            TRAIN_DIR = train_path
            break
if TRAIN_DIR is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if any(f.endswith('.jpg') or f.endswith('.png') for f in files):
            TRAIN_DIR = root
            break
if TRAIN_DIR is None:
    raise ValueError("Could not find any directory containing images. Did you add the dataset?")

print(f"Auto-detected training directory: {TRAIN_DIR}")

# ── VRAM FIX 1: Reduced Batch Size from 16 to 8 ──
train_dataset = DeepGlobeDataset(TRAIN_DIR, transform=get_train_transforms(256))
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)

model = DeepRoadSegModel(in_channels=3, base_ch=64, input_size=(256, 256), num_transformer_layers=12).to(device)
criterion = BCEDiceCLDiceLoss(bce_weight=0.1, dice_weight=0.2, cldice_weight=0.7)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ── VRAM FIX 2: Initialize Automatic Mixed Precision (AMP) Scaler ──
scaler = torch.cuda.amp.GradScaler(enabled=True)

num_epochs = 50
best_loss = float('inf')
patience = 6
patience_counter = 0

print(f"Found {len(train_dataset)} actual image-mask pairs! Starting training on {device}...")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    for images, masks in pbar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        
        # ── VRAM FIX 3: Forward pass with AMP enabled ──
        with torch.cuda.amp.autocast(enabled=True):
            logits = model(images, return_logits=True)
            loss = criterion(logits, masks)
            
        # ── VRAM FIX 4: Scaled Backward pass ──
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        
    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} Average Loss: {avg_loss:.4f}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(model.state_dict(), "/kaggle/working/best_deep_road_seg_model.pth")
        print(f"⭐ New best model found! Loss decreased to {best_loss:.4f}. Saved weights.")
    else:
        patience_counter += 1
        print(f"Loss did not improve from {best_loss:.4f} (Patience {patience_counter}/{patience})")
        if patience_counter >= patience:
            print(f"\n🛑 Early Stopping triggered! Loss plateaued for {patience} epochs.")
            break


Auto-detected training directory: /kaggle/input/datasets/balraj98/deepglobe-road-extraction-dataset/train
Found 6226 actual image-mask pairs! Starting training on cuda...


/tmp/ipykernel_24/1116734556.py:29: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=True)


Epoch 1/50:   0%|          | 0/779 [00:00<?, ?it/s]

/tmp/ipykernel_24/1116734556.py:50: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):


Epoch 1 Average Loss: 0.6814
⭐ New best model found! Loss decreased to 0.6814. Saved weights.


...

Epoch 48 Average Loss: 0.4276
Loss did not improve from 0.4260 (Patience 3/6)


Epoch 49/50:   0%|          | 0/779 [00:00<?, ?it/s]

Epoch 49 Average Loss: 0.4295
Loss did not improve from 0.4260 (Patience 4/6)


Epoch 50/50:   0%|          | 0/779 [00:00<?, ?it/s]

Epoch 50 Average Loss: 0.4254
⭐ New best model found! Loss decreased to 0.4254. Saved weights.
